# QueryBot v0.2 — Intent-gated resolution

Demonstrates the three-step flow that QueryBot uses internally:

```
record_intent → resolve_intent → compute_metrics
```

**Part 1** (no API key): `SpecResolver` used directly to show catalog validation — exact match, typo correction, wrong dimension kind, unknown metric.

**Part 2** (requires API key): Multi-turn `QueryBot` conversation. Each turn's trace shows the three mandatory steps before any analysis or final answer.

**Part 3** (Anthropic only): Prompt-cache efficiency. After turn 1 warms Layers A and B, subsequent turns report `cache_read_tokens > 0`.

### Prerequisites

1. Set your Anthropic API key: `export ANTHROPIC_API_KEY=sk-ant-...`
2. Install the agent extra: `pip install aitaem[agent-anthropic]`
3. Set `aitaem_folder_path` below to the path of your local `aitaem` repo.

In [ ]:
from __future__ import annotations

import os
import textwrap
from typing import Any

import pandas as pd

from aitaem.connectors import ConnectionManager
from aitaem.helpers import load_csvs_to_duckdb
from aitaem.specs import SpecCache
from aitaem.agent import (
    MetricIntent,
    QueryBot,
    QueryResponse,
    RunTrace,
    SpecMatchResult,
    SpecResolver,
    Status,
)

Set the path to the local folder which contains your AITAEM examples. The following sub-folders are needed inside `aitaem_folder_path`:
- `examples/data/`
- `examples/metrics/`
- `examples/slices/`
- `examples/segments/`

In [ ]:
aitaem_folder_path = "/path/to/aitaem"  # e.g. "/Users/you/Workspace/aitaem"

## Setup — load specs and connect to DuckDB

In [3]:
# Full catalog (metrics + slices + segments) for the SpecResolver demo.
# Loading segments here is fine — SpecResolver reads YAML metadata only
# and never queries the database.
spec_cache_full = SpecCache.from_yaml(
    metric_paths=aitaem_folder_path + "/examples/metrics/",
    slice_paths=aitaem_folder_path + "/examples/slices/",
    segment_paths=aitaem_folder_path + "/examples/segments/",
)
print(
    f"Full catalog: {len(spec_cache_full.metrics)} metrics, "
    f"{len(spec_cache_full.slices)} slices, "
    f"{len(spec_cache_full.segments)} segment(s)"
)
print(f"  Metrics  : {', '.join(spec_cache_full.metrics)}")
print(f"  Slices   : {', '.join(spec_cache_full.slices)}")
print(f"  Segments : {', '.join(spec_cache_full.segments)}")

Full catalog: 6 metrics, 3 slices, 1 segment(s)
  Metrics  : avg_revenue, campaign_count, ctr, max_revenue, roas, total_revenue
  Slices   : campaign_type, geo, industry
  Segments : platform


In [4]:
# QueryBot catalog excludes the platform segment: it references a dimension
# table (dim_platforms) absent from the example DuckDB.
spec_cache = SpecCache.from_yaml(
    metric_paths=aitaem_folder_path + "/examples/metrics/",
    slice_paths=aitaem_folder_path + "/examples/slices/",
)

# Create DuckDB from the bundled CSV if it doesn't already exist.
# The connection is opened in Part 2 — Part 1 (SpecResolver) needs no DB.
db_path = aitaem_folder_path + "/examples/data/ad_campaigns.duckdb"
if not os.path.exists(db_path):
    print("DuckDB file not found — creating from CSV …")
    load_csvs_to_duckdb(aitaem_folder_path + "/examples/data/ad_campaigns.csv", db_path)
    print(f"  Created {db_path}")
print("DuckDB path ready. Connection will open in Part 2.")

DuckDB path ready. Connection will open in Part 2.


### Helper functions

In [5]:
def _fmt_arg(v: Any) -> str:
    if isinstance(v, str) and len(v) > 14:
        return f"'{v[:10]}…'"
    if isinstance(v, list):
        return f"[{', '.join(repr(x) for x in v)}]"
    return repr(v)


def _print_resolution(label: str, result: SpecMatchResult) -> None:
    print(f"\n  {'─' * 62}")
    print(f"  {label}")
    print(f"  {'─' * 62}")
    if result.exact_match is not None:
        m = result.exact_match
        slices = ", ".join(m.slices) if m.slices else "(none)"
        print(f"  ✓  Exact match")
        print(f"     metric  : {m.metric_name}")
        print(f"     slices  : {slices}")
        print(f"     segment : {m.segment or '(none)'}")
        print(f"     token   : (minted by resolve_intent tool — empty here)")
    else:
        print(f"  ✗  No exact match — {len(result.near_misses)} near miss(es)")
        for nm in result.near_misses:
            line = f"     {nm.name!r:30s}  why_not={nm.why_not}"
            if nm.suggestions:
                line += f"  →  did you mean: {nm.suggestions}"
            print(line)


def _print_response(turn: int, question: str, response: QueryResponse) -> None:
    print(f"\n{'─' * 70}")
    print(f"Turn {turn}: {question}")
    print(f"{'─' * 70}")
    print(f"Status   : {response.status.value}")
    print(
        f"Narrative: {textwrap.fill(response.narrative, width=70, subsequent_indent='           ')}"
    )
    if response.status in (Status.error, Status.refused) and response.reason:
        print(f"Reason   : {response.reason}")
        return
    p = response.payload
    if p is None:
        return
    print(f"Metrics  : {', '.join(p.metrics_used) or '—'}")
    print(f"Slices   : {', '.join(p.slices_used) or '—'}")
    print(
        f"Period   : {p.period_type}"
        + (f"  {p.time_window[0]} → {p.time_window[1]}" if p.time_window else "")
    )
    if p.format_hints:
        hints = ", ".join(f"{k}={v}" for k, v in p.format_hints.items())
        print(f"Formats  : {hints}")
    print(f"Results  : {len(p.result_ids)} result(s)  primary={p.primary_result_id}")


def _print_sample(response: QueryResponse) -> None:
    p = response.payload
    if p is None or not p.sample:
        return
    df = pd.DataFrame(p.sample).dropna(axis=1, how="all")
    print(f"\nSample ({len(p.sample)} row(s)):")
    print(df.to_string(index=False))


def _print_trace(trace: RunTrace) -> None:
    print(
        f"\nTrace    : run={trace.run_id[:8]}  conv={trace.conversation_id[:8]}"
        f"  {trace.duration_ms:.0f}ms"
    )
    for tc in trace.tool_calls:
        non_null = {k: v for k, v in tc.args.items() if v is not None}
        args_str = ", ".join(f"{k}={_fmt_arg(v)}" for k, v in non_null.items())
        icon = "✓" if tc.success else "✗"
        print(f"  {icon} {tc.name}({args_str})")
    u = trace.usage
    cache_info = f"  cache_read={u.cache_read_tokens}" if u.cache_read_tokens else ""
    print(f"Tokens   : {u.input_tokens} in / {u.output_tokens} out{cache_info}")

---

## Part 1 — SpecResolver: catalog validation without an LLM

`SpecResolver` is the deterministic core behind `resolve_intent`.  It validates the LLM's proposed `metric_name`, `slices`, and `segment` against the catalog and returns either an `ExactMatch` or a list of `NearMiss` objects explaining what was wrong.

The four scenarios below cover the most common validation paths.

### Scenario 1 — Exact match

The LLM proposes `total_revenue` with slice `campaign_type`.  Both names exist in the catalog and `campaign_type` is registered as a slice (not a segment), so the resolver returns an `ExactMatch`.

In a live session the `resolve_intent` tool would mint a `spec_token` here (e.g. `sm_a3f1…`).  `SpecResolver` leaves it empty — token minting is the tool's job.

In [6]:
resolver = SpecResolver()

intent = MetricIntent(
    metric_concept="total revenue",   # free-text — LLM's interpretation
    scope="overall",
    period_type="all_time",
)

result = resolver.resolve(
    intent,
    proposed_metric_name="total_revenue",   # LLM's catalog guess
    proposed_slices=["campaign_type"],
    proposed_segment=None,
    spec_cache=spec_cache_full,
)
_print_resolution("Exact match: total_revenue sliced by campaign_type", result)


  ──────────────────────────────────────────────────────────────
  Exact match: total_revenue sliced by campaign_type
  ──────────────────────────────────────────────────────────────
  ✓  Exact match
     metric  : total_revenue
     slices  : campaign_type
     segment : (none)
     token   : (minted by resolve_intent tool — empty here)


### Scenario 2 — Typo: refused with user-nudge

`total_revenu` is not in the catalog.  The resolver runs `difflib.get_close_matches` (cutoff 0.75) and populates `NearMiss.suggestions` with the closest catalog name.

The LLM **does not auto-substitute** — even an obvious typo results in `status="refused"`.  Instead, the LLM includes the suggestion in its refusal message:

> *"I couldn't find a metric called `total_revenu`. Did you mean `total_revenue`?"*

The user confirms on the next turn, which then resolves to an exact match.  This conservative approach ensures no metric is silently swapped — the user always sees what is being computed.

In [7]:
intent = MetricIntent(
    metric_concept="total revenue",
    scope="overall",
    period_type="all_time",
)

result = resolver.resolve(
    intent,
    proposed_metric_name="total_revenu",   # typo: missing trailing 'e'
    proposed_slices=[],
    proposed_segment=None,
    spec_cache=spec_cache_full,
)
_print_resolution("Typo: 'total_revenu' → refused with user-nudge via suggestions", result)


  ──────────────────────────────────────────────────────────────
  Typo: 'total_revenu' → refused with user-nudge via suggestions
  ──────────────────────────────────────────────────────────────
  ✗  No exact match — 1 near miss(es)
     'total_revenu'                  why_not=unknown_metric  →  did you mean: ['total_revenue']


### Scenario 3 — Wrong dimension kind

`platform` is a **segment** spec, not a slice.  Passing it in `proposed_slices` gives `why_not="wrong_dimension_kind"` — the name exists in the catalog but in the wrong category.  The LLM learns to move it to `proposed_segment` instead.

In [8]:
intent = MetricIntent(
    metric_concept="CTR",
    scope="overall",
    period_type="all_time",
)

result = resolver.resolve(
    intent,
    proposed_metric_name="ctr",
    proposed_slices=["platform"],   # 'platform' is a segment spec, not a slice
    proposed_segment=None,
    spec_cache=spec_cache_full,
)
_print_resolution("Wrong dimension kind: 'platform' is a segment, not a slice", result)


  ──────────────────────────────────────────────────────────────
  Wrong dimension kind: 'platform' is a segment, not a slice
  ──────────────────────────────────────────────────────────────
  ✗  No exact match — 1 near miss(es)
     'platform'                      why_not=wrong_dimension_kind


### Scenario 4 — Unknown metric

`profit_margin` is not defined anywhere in the catalog and has no close match (difflib cutoff 0.75), so `suggestions` is empty.  The LLM must set `status="refused"` and prompt the user to define the metric or rephrase the question.

In [9]:
intent = MetricIntent(
    metric_concept="profit margin",
    scope="overall",
    period_type="all_time",
)

result = resolver.resolve(
    intent,
    proposed_metric_name="profit_margin",
    proposed_slices=[],
    proposed_segment=None,
    spec_cache=spec_cache_full,
)
_print_resolution("Unknown metric: 'profit_margin' (no close catalog match)", result)


  ──────────────────────────────────────────────────────────────
  Unknown metric: 'profit_margin' (no close catalog match)
  ──────────────────────────────────────────────────────────────
  ✗  No exact match — 1 near miss(es)
     'profit_margin'                 why_not=unknown_metric


---

## Part 2 — QueryBot: three-step flow in a real conversation

Each turn in the trace must show:
1. `record_intent` — LLM records what it understood the user to be asking for
2. `resolve_intent` — deterministic catalog validation; returns `spec_token` on success
3. `compute_metrics` — executes using the opaque `spec_token`; LLM cannot tamper with compute parameters

Analysis tools (`rank_by_value`, `period_over_period`, …) appear after step 3 when the question calls for them.

In [10]:
conn_mgr = ConnectionManager()
conn_mgr.add_connection("duckdb", path=db_path)

bot = QueryBot(
    model="anthropic:claude-haiku-4-5-20251001",
    spec_cache=spec_cache,
    connection_manager=conn_mgr,
)

### Turn 1 — multi-metric question

Two metrics in one question → two `record_intent` + `resolve_intent` + `compute_metrics` triplets, then a summary narrative.  This turn also **warms Layers A and B** in Anthropic's prompt cache.

In [11]:
response1 = await bot.chat("What was total revenue and ROAS across all campaigns?")
_print_response(1, "What was total revenue and ROAS across all campaigns?", response1)
_print_sample(response1)
_print_trace(response1.trace)


──────────────────────────────────────────────────────────────────────
Turn 1: What was total revenue and ROAS across all campaigns?
──────────────────────────────────────────────────────────────────────
Status   : ok
Narrative: Across all campaigns: - **Total Revenue**: $54,183,330.81 - **ROAS
           (Return on Ad Spend)**: 4.88 (meaning every $1 spent on ads
           generated $4.88 in revenue)
Metrics  : total_revenue, roas
Slices   : —
Period   : all_time
Results  : 2 result(s)  primary=cb859b15-b3b4-4296-8d6d-1aa6c2c82efb

Sample (1 row(s)):
period_type   metric_name slice_type slice_value segment_name segment_value  metric_value
   all_time total_revenue       none         all         none           all   54183330.81

Trace    : run=019f3ffd  conv=019f3ffd  5265ms
  ✓ record_intent(metric_concept='total revenue', scope='overall')
  ✓ record_intent(metric_concept='return on …', scope='overall')
  ✓ resolve_intent(intent_id=0, metric_name='total_revenue')
  ✓ resolve_intent(

### Turn 2 — breakdown with analysis

A breakdown question uses `rank_by_value` to surface the top campaign type after `compute_metrics` returns the full distribution.

In [12]:
response2 = await bot.chat("Which industry had the highest CTR?")
_print_response(2, "Which industry had the highest CTR?", response2)
_print_sample(response2)
_print_trace(response2.trace)


──────────────────────────────────────────────────────────────────────
Turn 2: Which industry had the highest CTR?
──────────────────────────────────────────────────────────────────────
Status   : ok
Narrative: **SaaS** had the highest click-through rate (CTR) at 4.08%,
           outperforming all other industries. The complete ranking by
           industry was: 1. SaaS: 4.08% 2. Fintech: 3.82% 3.
           E-commerce: 3.79% 4. Healthcare: 3.79% 5. EdTech: 3.77%
Metrics  : ctr
Slices   : industry
Period   : all_time
Results  : 1 result(s)  primary=7a594356-45e7-4365-b5b5-65320e106e37

Trace    : run=019f3ffd  conv=019f3ffd  12113ms
  ✓ record_intent(metric_concept='click-thro…', scope='subset', slice_type='industry')
  ✓ resolve_intent(intent_id=0, metric_name='ctr', slices=['industry'])
  ✓ compute_metrics(spec_token='sm_30a5d6b…')
  ✓ rank_by_value(result_id='8f09f347-c…', top_n=1)
  ✓ final_result(status='ok', narrative='**SaaS** h…', result_ids=['7a594356-45e7-4365-b5b5-65320e1

### Turn 3 — monthly trend

A `period_type="monthly"` question; `compute_metrics` returns one row per month and the LLM narrates the trend directly (no analysis tool needed for time series).

In [13]:
response3 = await bot.chat("How did total revenue change month-by-month in 2024?")
_print_response(3, "How did total revenue change month-by-month in 2024?", response3)
_print_sample(response3)
_print_trace(response3.trace)


──────────────────────────────────────────────────────────────────────
Turn 3: How did total revenue change month-by-month in 2024?
──────────────────────────────────────────────────────────────────────
Status   : ok
Narrative: Total revenue in 2024 showed month-to-month volatility with both
           growth and decline periods:  **Month-by-Month Changes:** -
           **January**: $4,691,994 (baseline) - **February**:
           $4,749,203 (+1.2%) - **March**: $4,407,519 (-7.2%) -
           **April**: $5,143,464 (+16.7%) — strong recovery - **May**:
           $3,768,913 (-26.7%) — significant decline - **June**:
           $4,362,176 (data shows improvement trend starting) -
           **July**: (continuing through year) - **September**:
           $4,749,430 - **November**: $4,484,003 - **December**:
           $5,159,353  The year exhibited significant swings, with the
           largest decline occurring in May (-26.7% from April) and
           strong recovery in April (+16.7

---

## Part 3 — Prompt-cache efficiency (Anthropic only)

QueryBot v0.2 divides the system prompt into three layers:

| Layer | Content | Cached? |
|-------|---------|--------|
| A | Workflow rules and tool guidance (identical for all tenants) | Yes — `anthropic_cache_instructions="5m"` |
| B | Per-tenant metric/slice/segment catalog | Yes — same cache breakpoint |
| C | Today's date (changes each calendar day) | No — dynamic instruction |

After turn 1, Layers A+B are stored in Anthropic's prompt cache.  Turns 2 and 3 should read them from cache, reducing both latency and input token cost.  The metric is `cache_read_tokens` in the usage block.

In [14]:
def _cache_summary(label: str, response: Any) -> None:
    u = response.trace.usage
    ct = u.cache_read_tokens or 0
    status = "✓  served from cache" if ct > 0 else "✗  no cache hit"
    print(f"{label}")
    print(f"  input_tokens       : {u.input_tokens:>6}")
    print(f"  cache_read_tokens  : {ct:>6}  {status}")
    print(f"  output_tokens      : {u.output_tokens:>6}")
    print()


if "anthropic:" in bot._model:
    print("Cache efficiency summary (Anthropic only)")
    print("─" * 50)
    _cache_summary("Turn 1 (warms cache — no read expected)", response1)
    _cache_summary("Turn 2 (Layers A+B should be cached)", response2)
    _cache_summary("Turn 3 (Layers A+B should be cached)", response3)
else:
    print(f"Cache check skipped — provider is not Anthropic (model={bot._model!r}).")

Cache efficiency summary (Anthropic only)
──────────────────────────────────────────────────
Turn 1 (warms cache — no read expected)
  input_tokens       :  20571
  cache_read_tokens  :  12312  ✓  served from cache
  output_tokens      :    535

Turn 2 (Layers A+B should be cached)
  input_tokens       :  34925
  cache_read_tokens  :  20520  ✓  served from cache
  output_tokens      :    499

Turn 3 (Layers A+B should be cached)
  input_tokens       :  45429
  cache_read_tokens  :  20520  ✓  served from cache
  output_tokens      :    631

